# Healthcare Reducto OracleDB Demo Notebook

This notebook demonstrates the healthcare version of the Reducto + Oracle flow:

1. Load `.env`.
2. Connect to Oracle and ensure the schema exists.
3. Inspect documents tagged with `filing_type = 'HEALTHCARE'`.
4. Preview extracted tables for labs, vitals, medications, or schedules.
5. Retrieve chunk evidence without requiring a live embedding call.
6. Optionally ingest a healthcare document URL through Reducto.

## Platform Context

![Executive thesis: Reducto AI + Oracle Database](assets/reducto_oracle_executive_thesis.svg)

![Capability map: Oracle Database approach vs standalone vector platforms](assets/reducto_oracle_platform_capability_map.svg)

![Operating model and risk controls](assets/reducto_oracle_operating_model_risk_controls.svg)


In [1]:
import os
import sys
from pathlib import Path
from dataclasses import replace


def find_sdk_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "src" / "reducto" / "lib" / "oracledb").exists():
            return path
    raise RuntimeError("Could not locate the reducto-python-sdk repository root")


ROOT = find_sdk_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


def load_env(path=ROOT / "examples" / "oracledb" / ".env"):
    if not path.exists():
        return
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        if line.startswith("export "):
            line = line[len("export ") :].strip()
        key, value = line.split("=", 1)
        os.environ[key.strip()] = value.strip().strip("'\"")


def optional_int(name):
    value = os.getenv(name)
    return int(value) if value else None


load_env()

DOMAIN_LABEL = "HEALTHCARE"
DOMAIN_ENTITY = os.getenv("HEALTHCARE_ORG", "DEMO_HEALTH")
DOMAIN_YEAR = optional_int("HEALTHCARE_YEAR")
QUESTION = os.getenv(
    "HEALTHCARE_DEMO_QUESTION",
    "What diagnoses, medications, follow-up instructions, or abnormal results are documented?",
)

print("Oracle user:", os.getenv("ORACLE_USER"))
print("Oracle DSN:", os.getenv("ORACLE_DSN"))
print("Domain Entity:", DOMAIN_ENTITY)
print("Reducto key set:", bool(os.getenv("REDUCTO_API_KEY")))
print("Cohere key set:", bool(os.getenv("CO_API_KEY")))
print("Healthcare ingest enabled:", os.getenv("RUN_HEALTHCARE_INGEST") == "1")
print("Healthcare source URI set:", bool(os.getenv("HEALTHCARE_SOURCE_URI")))
print("Domain label:", DOMAIN_LABEL)

Oracle user: REDUCTO_RAG_APP
Oracle DSN: localhost:1521/FREEPDB1
Domain Entity: MEDLINEPLUS
Reducto key set: True
Cohere key set: True
Healthcare ingest enabled: True
Healthcare source URI set: True
Domain label: HEALTHCARE


## Connect to Oracle and inspect the schema

In [2]:
from reducto.lib.oracledb.config import connect_oracle, vector_dimensions_from_env
from reducto.lib.oracledb.oracle import OracleSchemaManager

connection = connect_oracle()
OracleSchemaManager(connection).create_schema(vector_dimensions=vector_dimensions_from_env())
print("Oracle schema is ready")

for table in ["DOCUMENTS", "DOCUMENT_CHUNKS", "PARSED_TABLES", "FINANCIAL_FACTS"]:
    with connection.cursor() as cursor:
        cursor.execute(f"select count(*) from {table}")
        print(table, cursor.fetchone()[0])

Oracle schema is ready
DOCUMENTS 1
DOCUMENT_CHUNKS 1
PARSED_TABLES 47
FINANCIAL_FACTS 1222


## Ingest a healthcare URL

Set `RUN_HEALTHCARE_INGEST=1` and `HEALTHCARE_SOURCE_URI=<url>` before running this cell. The notebook keeps parsed tables but clears `financial_facts`, because those facts are tuned for SEC-style financial statements.

In [ ]:
RUN_INGEST = os.getenv("RUN_HEALTHCARE_INGEST") == "1"
SOURCE_URI = os.getenv("HEALTHCARE_SOURCE_URI", "")

if not RUN_INGEST:
    print("Skipping ingest. Set RUN_HEALTHCARE_INGEST=1 and HEALTHCARE_SOURCE_URI to parse a document.")
elif not SOURCE_URI:
    raise RuntimeError("HEALTHCARE_SOURCE_URI must be set when RUN_HEALTHCARE_INGEST=1")
else:
    from reducto.lib.oracledb.models import DocumentMetadata
    from reducto.lib.oracledb.oracle import OracleDocumentRepository
    from reducto.lib.oracledb.embeddings import embedding_provider_from_env
    from reducto.lib.oracledb.reducto_client import ReductoDocumentParser

    parse_result = ReductoDocumentParser().parse_url(SOURCE_URI)
    parse_result = replace(parse_result, financial_facts=[])
    metadata = DocumentMetadata(
        source_uri=SOURCE_URI,
        source_kind="url",
        company=DOMAIN_ENTITY,
        fiscal_year=DOMAIN_YEAR,
        filing_type=DOMAIN_LABEL,
        title=os.getenv("HEALTHCARE_TITLE", "Healthcare Reducto demo document"),
    )
    document_id = OracleDocumentRepository(connection).store_parse_result(
        metadata,
        parse_result,
        embedding_provider_from_env(input_type="search_document"),
    )
    print("Stored healthcare document", document_id)
    print("Chunks:", len(parse_result.chunks), "Tables:", len(parse_result.tables))

## List stored healthcare documents

In [5]:
def fetch_domain_documents(limit=10):
    with connection.cursor() as cursor:
        cursor.execute(
            f"""
            select document_id, company, fiscal_year, filing_type, title, source_uri
            from documents
            where upper(filing_type) = upper(:domain_label)
            order by created_at desc
            fetch first {max(1, int(limit))} rows only
            """,
            domain_label=DOMAIN_LABEL,
        )
        return cursor.fetchall()


domain_docs = fetch_domain_documents()
if not domain_docs:
    if os.getenv("RUN_HEALTHCARE_INGEST") == "1":
        print(
            "No HEALTHCARE documents found yet. The ingest cell is later in this notebook; run that cell, then rerun this listing/evidence section."
        )
    else:
        print("No HEALTHCARE documents found yet. Use the optional ingest cell below to add one.")
else:
    for row in domain_docs:
        print(row)

(21, 'MEDLINEPLUS', 2024, 'HEALTHCARE', 'MedlinePlus Metformin Drug Information', 'https://medlineplus.gov/druginfo/meds/a696005.html')


## Preview extracted healthcare tables

For healthcare documents, tables are useful for lab panels, vitals, medication lists, encounter timelines, and billing/schedule grids. The financial-fact promotion table is intentionally not used here.

In [6]:
from reducto.lib.oracledb.utils import read_lob

if not domain_docs:
    print("Skipping table preview because no HEALTHCARE documents are stored yet.")
else:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            select d.document_id, d.title, t.table_index, t.page_number, t.raw_content
            from parsed_tables t
            join documents d on d.document_id = t.document_id
            where upper(d.filing_type) = upper(:domain_label)
            order by d.created_at desc, t.table_index
            fetch first 5 rows only
            """,
            domain_label=DOMAIN_LABEL,
        )
        table_rows = cursor.fetchall()

    if not table_rows:
        print("No parsed healthcare tables found. Chunk search can still work without tables.")
    for document_id, title, table_index, page_number, raw_content in table_rows:
        text = str(read_lob(raw_content) or "")
        print(f"Document {document_id} | table {table_index} | page {page_number} | {title}")
        print(text[:700].replace("\n", " "))
        print("---")

No parsed healthcare tables found. Chunk search can still work without tables.


## Retrieve healthcare chunk evidence

This cell uses a lightweight SQL keyword search so the notebook can run without a live embedding request. Set `RUN_LIVE_EMBED_SEARCH=1` in your environment to run semantic search in the next cell.

In [7]:
from reducto.lib.oracledb.qa import format_answer, answer_from_search_results
from reducto.lib.oracledb.models import SearchResult

SEARCH_TERMS = ["diagnosis", "medication", "follow-up", "discharge", "assessment", "lab"]


def lexical_chunk_search(terms, limit=5):
    clauses = []
    params = {"domain_label": DOMAIN_LABEL}
    for index, term in enumerate(terms):
        key = f"term_{index}"
        clauses.append(f"lower(c.content) like :{key}")
        params[key] = f"%{term.lower()}%"
    term_sql = " or ".join(clauses)
    with connection.cursor() as cursor:
        cursor.execute(
            f"""
            select d.document_id, c.chunk_id, c.content, d.company, d.fiscal_year,
                   d.filing_type, c.page_start, c.page_end, d.source_uri
            from document_chunks c
            join documents d on d.document_id = c.document_id
            where upper(d.filing_type) = upper(:domain_label)
              and ({term_sql})
            order by c.created_at desc
            fetch first {max(1, int(limit))} rows only
            """,
            params,
        )
        rows = cursor.fetchall()

    results = []
    for row in rows:
        content = str(read_lob(row[2]) or "")
        score = sum(1 for term in terms if term.lower() in content.lower())
        results.append(
            SearchResult(
                document_id=int(row[0]),
                chunk_id=int(row[1]),
                score=float(score),
                content=content,
                company=row[3],
                fiscal_year=int(row[4]) if row[4] is not None else None,
                filing_type=row[5],
                page_start=int(row[6]) if row[6] is not None else None,
                page_end=int(row[7]) if row[7] is not None else None,
                source_uri=str(row[8] or ""),
            )
        )
    return results


if not domain_docs:
    print("Skipping evidence retrieval because no HEALTHCARE documents are stored yet.")
else:
    results = lexical_chunk_search(SEARCH_TERMS, limit=5)
    if not results:
        print("No keyword matches found for:", SEARCH_TERMS)
    else:
        print(format_answer(answer_from_search_results(QUESTION, results, evidence_limit=3)))

Question
  What is metformin used for, and what precautions are mentioned?

Answer
  If the victim has collapsed, had a seizure, has trouble breathing, or can't be awakened, immediately call emergency services at 911.# Metformin (cont.) ## Symptoms of overdose may include hypoglycemia symptoms as well as the following: • - extreme tiredness • - weakness • - discomfort • - vomiting • - nausea • - stomach pain • - decreased appetite • - deep, rapid breathing • - shortness of breath • - dizziness • - lightheadedness • - abnormally fast or slow heartbeat • - muscle pain • - feeli...

Evidence
  1. page 7
     Place the medication in a safe location – one that is up and away and out of their sight and reach. https:// www.upandaway.org# Metformin (cont.) ## In case of emergency/overdose In case of overdose, call the poison control helpline at 1-800-222-1222. Information is also available online at https://www.poisonhelp.org/help . If the victim has collapsed, had a seizure, has trouble breat

## Optional semantic search

This calls the configured embedding provider, so it is opt-in.

In [8]:
RUN_LIVE_EMBED_SEARCH = os.getenv("RUN_LIVE_EMBED_SEARCH") == "1"

if not RUN_LIVE_EMBED_SEARCH:
    print("Skipping live semantic search. Set RUN_LIVE_EMBED_SEARCH=1 to enable it.")
elif not domain_docs:
    print("Skipping semantic search because no HEALTHCARE documents are stored yet.")
else:
    from reducto.lib.oracledb.models import SearchFilters
    from reducto.lib.oracledb.retrieval import OracleHybridRetriever
    from reducto.lib.oracledb.embeddings import embedding_provider_from_env

    retriever = OracleHybridRetriever(
        connection,
        embedding_provider_from_env(input_type="search_query"),
    )
    semantic_results = retriever.semantic_search(
        QUESTION,
        filters=SearchFilters(filing_type=DOMAIN_LABEL),
        limit=5,
    )
    print(format_answer(answer_from_search_results(QUESTION, semantic_results, evidence_limit=3)))

Question
  What is metformin used for, and what precautions are mentioned?

Answer
  If you experience any of the following symptoms, stop taking metformin and call your doctor immediately: extreme tiredness, weakness, or discomfort; nausea; vomiting; stomach pain; decreased appetite; deep and rapid breathing or shortness of breath; dizziness; lightheadedness; fast or slow heartbeat; muscle pain; or feeling cold, especially in your hands or feet.

Evidence
  1. page 3
     Tell your doctor if you regularly drink alcohol or sometimes drink large amounts of alcohol in a short time (binge drinking). Drinking alcohol increases your risk of developing lactic acidosis or may cause a decrease in blood sugar. Ask your doctor how much alcohol is safe to drink while you are taking metformin. Keep all appointments with your doctor and the laboratory. Your doctor will order certain tests before and during treatment to check how well your kidneys are working and your body's response to metformin. T

In [12]:
connection.close()
print("Connection closed")

Connection closed
